Ensure runtime is 2025.10
  or else it breaks


In [ ]:
!git clone https://github.com/uds-lsv/GermEval-2018-Data.git

Cloning into 'GermEval-2018-Data'...
remote: Enumerating objects: 60, done.
remote: Total 60 (delta 0), reused 0 (delta 0), pack-reused 60 (from 1)
Receiving objects: 100% (60/60), 14.78 MiB | 17.78 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [ ]:
file_path = "GermEval-2018-Data/germeval2018.training.txt" # Assuming this is the desired file

parsed_data = []
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) >= 2:
                fine_grained_class = parts[-1]
                coarse_grained_class = parts[-2]
                tweet_text = " ".join(parts[:-2]) # Reconstruct tweet text
                parsed_data.append({
                    "tweet_text": tweet_text,
                    "coarse_grained_class": coarse_grained_class,
                    "fine_grained_class": fine_grained_class
                })
            else:
                print(f"Skipping line due to insufficient parts: {line}")

    print(f"Successfully parsed {len(parsed_data)} lines.")
    print("First 5 parsed entries:")
    for entry in parsed_data[:5]:
        print(entry)

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully parsed 5009 lines.
First 5 parsed entries:
{'tweet_text': '@corinnamilborn Liebe Corinna, wir würden dich gerne als Moderatorin für uns gewinnen! Wärst du begeisterbar?', 'coarse_grained_class': 'OTHER', 'fine_grained_class': 'OTHER'}
{'tweet_text': '@Martin28a Sie haben ja auch Recht. Unser Tweet war etwas missverständlich. Dass das BVerfG Sachleistungen nicht ausschließt, kritisieren wir.', 'coarse_grained_class': 'OTHER', 'fine_grained_class': 'OTHER'}
{'tweet_text': '@ahrens_theo fröhlicher gruß aus der schönsten stadt der welt theo ⚓️', 'coarse_grained_class': 'OTHER', 'fine_grained_class': 'OTHER'}
{'tweet_text': '@dushanwegner Amis hätten alles und jeden gewählt...nur Hillary wollten sie nicht und eine Fortsetzung von Obama-Politik erst recht nicht..!', 'coarse_grained_class': 'OTHER', 'fine_grained_class': 'OTHER'}
{'tweet_text': '@spdde kein verläßlicher Verhandlungspartner. Nachkarteln nach den Sondierzngsgesprächen - schickt diese Stümper #SPD in die Versenkung.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import json
import os
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
import torch
import random
import time
import datetime
from torch import nn
import argparse
from torch.optim import AdamW

In [ ]:
df = pd.DataFrame(parsed_data)
np.unique(df['coarse_grained_class'], return_counts=True)

(array(['OFFENSE', 'OTHER'], dtype=object), array([1688, 3321]))

In [ ]:
3321/(3321+1688)

0.6630065881413456

In [ ]:
df['coarse_grained_class'] = df['coarse_grained_class'].map({'OFFENSE': 1, 'OTHER': 0})

In [ ]:
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

In [ ]:
y = df['coarse_grained_class']
X = df[['tweet_text']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.5 MB/s eta 0:00:00


In [ ]:
import emoji
import re

def handle_emojis(text):
    if not isinstance(text, str):
        return text

    text = emoji.demojize(text, delimiters=(" EMOJI_", " "))
    text = re.sub(r'EMOJI_[a-z_]+', 'EMOJI', text)
    return text


# ----------------------------
# AFTER TRAIN / TEST SPLIT
# ----------------------------
X_train['tweet_text'] = X_train['tweet_text'].apply(handle_emojis)
X_test['tweet_text'] = X_test['tweet_text'].apply(handle_emojis)


In [ ]:
y_test

,coarse_grained_class
777,0
3735,1
186,0
3779,0
2958,1
...,...
2248,1
3311,0
1452,0
308,0


In [ ]:
!git clone https://github.com/ulyavedenina/AI-Generated-German-Hate-Speech-Detector.git

Cloning into 'AI-Generated-German-Hate-Speech-Detector'...
remote: Enumerating objects: 379, done.
remote: Counting objects: 100% (379/379), done.
remote: Compressing objects: 100% (257/257), done.
remote: Total 379 (delta 207), reused 263 (delta 117), pack-reused 0 (from 0)
Receiving objects: 100% (379/379), 13.72 MiB | 8.82 MiB/s, done.
Resolving deltas: 100% (207/207), done.


In [ ]:
BATCH_SIZE = 64
lr = 2e-5
NUM_EPOCHS = 2
model_id = 'dbmdz/bert-base-german-uncased'
#model_id = 'deepset/gbert-large'

# Set up output directory
output_dir = "output"
print(output_dir)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
print("Model will be saved to %s" % output_dir)

# Tokenize using BERT tokenizer
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained(model_id, do_lower_case=True)

# X_train and X_test are now DataFrames with converted emojis from cell eiDeckFXy1nN.
# We can directly extract their 'tweet_text' column values.
X_train_list = X_train['tweet_text'].values
X_test_list = X_test['tweet_text'].values

y_train = y_train.values
y_test = y_test.values

# Encode data
def encode(comment, labels):
    input_ids = []
    attention_masks = []

    for word in comment:
        encoded_dict = tokenizer.encode_plus(
            word,
            add_special_tokens=True,
            max_length=256,
            padding='max_length', # Changed from pad_to_max_length=True
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)

    dataset = TensorDataset(input_ids, attention_masks, labels)

    return dataset

X_train = encode(X_train_list, y_train)
X_test = encode(X_test_list, y_test)

train_dataloader = DataLoader(X_train, sampler=SequentialSampler(X_train), batch_size=BATCH_SIZE)
test_dataloader = DataLoader(X_test, sampler=SequentialSampler(X_test), batch_size=BATCH_SIZE)

# Load BERT model
model = BertForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False,
)
params = list(model.named_parameters())

# Set up optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=lr, eps=1e-8)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_dataloader) * 2)

# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

# Function to calculate the accuracy of our predictions vs labels
def f_score(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return f1_score(labels_flat, pred_flat)

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

# Training loop
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")
model.to(device)

training_stats = []
all_gold_labels = []
all_predictions = []
total_t0 = time.time()

# Train for each epoch
for epoch_i in range(0, NUM_EPOCHS):
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, NUM_EPOCHS))
    print('Training...')

    t0 = time.time()
    total_train_loss = 0
    model.train()

    for step, batch in enumerate(train_dataloader):
        if step % 40 == 0 and not step == 0:
            elapsed = format_time(time.time() - t0)
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()
        result = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels, return_dict=True)

        loss = result.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    training_time = format_time(time.time() - t0)

    print("")
    print("  Average training loss: {0:.3f}".format(avg_train_loss))
    print("  Training epoch took: {:}".format(training_time))

    # Validation
    print("")
    print("Running Validation...")

    t0 = time.time()
    model.eval()
    all_predictions = []
    all_gold_labels = []
    all_texts = []

    total_eval_accuracy = 0
    total_eval_loss = 0
    nb_eval_steps = 0
    total_f1_score = 0

    wrong_predictions = []

    for batch in test_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with torch.no_grad():
            result = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels, return_dict=True)

        loss = result.loss
        logits = result.logits




        preds = torch.argmax(logits, dim=1)
      #test adding
        for i in range(len(preds)):
          if preds[i].item() != b_labels[i].item():
              wrong_predictions.append({
                  "text": tokenizer.decode(b_input_ids[i], skip_special_tokens=True),
                  "true_label": b_labels[i].item(),
                  "pred_label": preds[i].item()
              })




        total_eval_loss += loss.item()
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        all_gold_labels.extend(label_ids)
        all_predictions.extend(np.argmax(logits, axis=1))
        all_texts.extend([tokenizer.decode(ids, skip_special_tokens=True) for ids in b_input_ids])

        total_eval_accuracy += flat_accuracy(logits, label_ids)
        total_f1_score += f_score(logits, label_ids)

    all_predictions = np.array(all_predictions)
    all_gold_labels = np.array(all_gold_labels)

    np.savetxt('validation_predictions.txt', np.column_stack((all_gold_labels, all_predictions, all_texts)), fmt='%s', delimiter='\t', header='gold\tpred\ttext', comments='')

    avg_val_accuracy = total_eval_accuracy / len(test_dataloader)
    print("  Accuracy: {0:.3f}".format(avg_val_accuracy))

    avg_val_fscore = total_f1_score / len(test_dataloader)
    print("  F-Score: {0:.3f}".format(avg_val_fscore))

    avg_val_loss = total_eval_loss / len(test_dataloader)

    validation_time = format_time(time.time() - t0)

    print("  Validation Loss: {0:.3f}".format(avg_val_loss))
    print("  Validation took: {:}".format(validation_time))

    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Valid. F-score.': avg_val_fscore,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )

print("")
print("Training complete!")

model_to_save = model.module if hasattr(model, 'module') else model
model_to_save.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

with open(f'{output_dir}/training_stats.json', 'w') as json_file:
    json.dump(training_stats, json_file)

print("Training stats saved to 'training_stats.json'")
print("Predictions saved to 'validation_predictions.txt'")
df_wrong = pd.DataFrame(wrong_predictions)
df_wrong.to_csv(f"{output_dir}/wrong_predictions.csv", index=False, encoding="utf-8")
print(f"Wrong predictions saved to '{output_dir}/wrong_predictions.csv'")

output
Model will be saved to output
Loading BERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dbmdz/bert-base-german-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using cuda device

======== Epoch 1 / 2 ========
Training...
  Batch    40  of     63.    Elapsed: 0:01:41.

  Average training loss: 0.569
  Training epoch took: 0:02:40

Running Validation...
  Accuracy: 0.772
  F-Score: 0.554
  Validation Loss: 0.476
  Validation took: 0:00:17

======== Epoch 2 / 2 ========
Training...
  Batch    40  of     63.    Elapsed: 0:01:42.

  Average training loss: 0.421
  Training epoch took: 0:02:41

Running Validation...
  Accuracy: 0.802
  F-Score: 0.678
  Validation Loss: 0.415
  Validation took: 0:00:17

Training complete!
Total training took 0:05:57 (h:mm:ss)
Training stats saved to 'training_stats.json'
Predictions saved to 'validation_predictions.txt'
Wrong predictions saved to 'output/wrong_predictions.csv'


In [ ]:
# Function to extract embeddings (CLS token representation)
def get_embeddings_cls(dataloader, model, device):
    embeddings = []
    model.eval()  # Set the model to evaluation mode

    for batch in dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)

        with torch.no_grad():
            # Access the base BERT model within the BertForSequenceClassification
            # This returns (last_hidden_state, pooler_output) if output_hidden_states=False
            outputs = model.bert(b_input_ids, attention_mask=b_input_mask)
            last_hidden_states = outputs[0]  # Get the last hidden states

            # Extract the [CLS] token embedding (first token in the sequence)
            cls_embeddings = last_hidden_states[:, 0, :]

        embeddings.append(cls_embeddings.cpu().numpy())

    return np.concatenate(embeddings, axis=0)

In [ ]:
# Function to extract embeddings (average of all token representations)
def get_embeddings_avg(dataloader, model, device):
    embeddings = []
    model.eval()  # Set the model to evaluation mode

    for batch in dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)

        with torch.no_grad():
            # Access the base BERT model within the BertForSequenceClassification
            outputs = model.bert(b_input_ids, attention_mask=b_input_mask)
            last_hidden_states = outputs[0]  # Get the last hidden states (batch_size, seq_len, hidden_size)

            # Average all token embeddings, excluding padding tokens
            # Expand attention_mask to match the hidden_states dimensions for element-wise multiplication
            masked_output = last_hidden_states * b_input_mask.unsqueeze(-1) # Zero out padded token embeddings
            sum_embeddings = torch.sum(masked_output, 1) # Sum along the sequence length dimension

            # Calculate the number of non-padded tokens for each sequence
            num_tokens = torch.sum(b_input_mask, 1).unsqueeze(-1) # (batch_size, 1)

            # Compute the average, handling potential division by zero for empty sequences (though unlikely here)
            # Clamp to prevent division by zero for sequences where num_tokens might be 0 (e.g., if attention_mask is all zeros)
            avg_embeddings = sum_embeddings / torch.clamp(num_tokens, min=1e-9)

        embeddings.append(avg_embeddings.cpu().numpy())

    return np.concatenate(embeddings, axis=0)

In [ ]:
print("Generating CLS embeddings for training set...")
train_cls_embeddings = get_embeddings_cls(train_dataloader, model, device)
print(f"Shape of training CLS embeddings: {train_cls_embeddings.shape}")

print("Generating CLS embeddings for test set...")
test_cls_embeddings = get_embeddings_cls(test_dataloader, model, device)
print(f"Shape of test CLS embeddings: {test_cls_embeddings.shape}")

print("Generating mean-pooled embeddings for training set...")
train_avg_embeddings = get_embeddings_avg(train_dataloader, model, device)
print(f"Shape of training mean embeddings: {train_avg_embeddings.shape}")

print("Generating mean-pooled embeddings for test set...")
test_avg_embeddings = get_embeddings_avg(test_dataloader, model, device)
print(f"Shape of test mean embeddings: {test_avg_embeddings.shape}")

# Save all embeddings as .npy files
np.save(os.path.join(output_dir, "X_train_bert_cls.npy"), train_cls_embeddings)
np.save(os.path.join(output_dir, "X_test_bert_cls.npy"), test_cls_embeddings)

np.save(os.path.join(output_dir, "X_train_bert_mean.npy"), train_avg_embeddings)
np.save(os.path.join(output_dir, "X_test_bert_mean.npy"), test_avg_embeddings)

print("All BERT embedding files generated and saved.")


Generating CLS embeddings for training set...
Shape of training CLS embeddings: (4007, 768)
Generating CLS embeddings for test set...
Shape of test CLS embeddings: (1002, 768)
Generating mean-pooled embeddings for training set...
Shape of training mean embeddings: (4007, 768)
Generating mean-pooled embeddings for test set...
Shape of test mean embeddings: (1002, 768)
All BERT embedding files generated and saved.


In [ ]:
train_cls_embeddings[0]

array([ 6.93914518e-02,  4.22213078e-01, -6.75857782e-01,  1.07172513e+00,
       -2.15062052e-01,  9.17508781e-01,  1.11157131e+00, -1.03272736e+00,
        7.48405337e-01,  1.36229026e+00, -5.58107078e-01, -6.64625168e-01,
       -5.63615382e-01, -8.92656371e-02,  4.89485040e-02, -3.75308156e-01,
        4.38436508e-01, -3.16790044e-01, -4.76989865e-01, -3.75281125e-01,
        4.04380411e-01, -5.79705477e-01, -3.32786411e-01, -6.21946812e-01,
        4.04950023e-01,  1.04654573e-01, -1.27188757e-01, -6.09161258e-01,
        5.90023458e-01,  2.35950872e-01, -4.93821651e-01, -1.45373613e-01,
        2.32345596e-01, -8.57838020e-02,  9.70234215e-01, -9.32359338e-01,
        8.94485950e-01,  3.15905213e-01, -7.87997901e-01,  1.07292473e+00,
        1.86909720e-01,  1.28597343e+00, -1.09430633e-01, -2.87693739e-01,
        1.58629060e-01, -1.27756882e+00,  3.26026278e-03, -1.04230940e+00,
        7.40893409e-02, -3.82359803e-01, -1.75740048e-01, -8.32512796e-01,
       -1.09254026e+00,  

In [ ]:
train_embeddings =train_cls_embeddings
test_embeddings = test_cls_embeddings

In [ ]:
import regex

ignore_chars = {' ', '`', '^','a','A'}

def emoji_tokenizer(text, remove_stopwords=False):
    tokens = regex.findall(r'\p{L}+\p{M}*|[\p{So}\p{Sk}]', text)
    tokens = [t for t in tokens if t not in ignore_chars]
    if remove_stopwords:
        tokens = [t for t in tokens if t.lower() not in german_stopwords]
    return tokens

def emoji_tokenizer_agg(text, remove_stopwords=False):
    tokens = regex.findall(r'\p{L}+\p{M}*|[\p{So}\p{Sk}]', text)
    tokens = ['EMOJI' if regex.match(r'[\p{So}\p{Sk}]', t) else t for t in tokens if t not in ignore_chars]
    if remove_stopwords:
        tokens = [t for t in tokens if t.lower() not in german_stopwords]
    return tokens

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer
N= 4400
vectorizer = CountVectorizer(max_features=N,tokenizer=lambda text: emoji_tokenizer(text, remove_stopwords=False),lowercase=False)
#vectorizer = CountVectorizer(max_features=N)
#vectorizer = CountVectorizer(max_features=N)
# Fit and transform the tweet_text data
X = vectorizer.fit_transform(X_train_list)

print(f"Shape of the vectorized data: {X.shape}")
print(f"Number of features (unique words): {len(vectorizer.get_feature_names_out())}")
print("Top 10 features (words):", vectorizer.get_feature_names_out()[:10])


# Vectorize test data using the SAME vectorizer
X_test = vectorizer.transform(X_test_list)


Shape of the vectorized data: (4007, 4400)
Number of features (unique words): 4400
Top 10 features (words): ['AFD' 'AFDtrischi' 'AG' 'ALLES' 'AN' 'APatzwahl' 'APosener' 'ARD'
 'ARDKontraste' 'ARDde']


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
from scipy.sparse import hstack, csr_matrix

# Convert embeddings to sparse
train_embeddings_sparse = csr_matrix(train_embeddings)
test_embeddings_sparse  = csr_matrix(test_embeddings)

# Merge features
X_train_combined = hstack([X, train_embeddings_sparse])
X_test_combined  = hstack([X_test, test_embeddings_sparse])

print(X_train_combined.shape)
print(X_test_combined.shape)


(4007, 5168)
(1002, 5168)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

rf_classifier = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
#NEW
#rf_classifier.fit(train_embeddings, y_train)
#y_pred = rf_classifier.predict(test_embeddings)
#OLD
rf_classifier.fit(X_train_combined, y_train)
y_pred = rf_classifier.predict(X_test_combined)


accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1: {f1:.4f}")

Accuracy: 0.8064
F1: 0.6901


In [ ]:
y_test

array([0, 1, 0, ..., 0, 0, 0])

In [ ]:
# ---- SAVE WRONG ANSWERS TO CSV ----

wrong_rows = []

for text, true, pred in zip(X_test_list, y_test, y_pred):
        #if true != pred:
        wrong_rows.append({
            "text": text,
            "true_label": true,
            "predicted_label": pred
        })

df_wrong = pd.DataFrame(wrong_rows)
df_wrong.to_csv("both_Pred.csv", index=False, encoding="utf-8")

In [ ]:
# ONLY LOOKING AT THE CLS TOKEN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import os

# Initialize and train the Random Forest Classifier
print("Training Random Forest Classifier...")
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 uses all available cores
rf_classifier.fit(train_embeddings, y_train)
print("Random Forest Classifier training complete.")

# Make predictions on the test set
print("Making predictions on the test set...")
rf_predictions = rf_classifier.predict(test_embeddings)

# Evaluate the Random Forest model
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions)

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest F1-Score: {rf_f1:.4f}")

Training Random Forest Classifier...
Random Forest Classifier training complete.
Making predictions on the test set...
Random Forest Accuracy: 0.7894
Random Forest F1-Score: 0.6645


In [ ]:
# AVERAGING ALL OF THE WORDS TOGETHER
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import os

# Initialize and train the Random Forest Classifier
print("Training Random Forest Classifier...")
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 uses all available cores
rf_classifier.fit(train_embeddings, y_train)
print("Random Forest Classifier training complete.")

# Make predictions on the test set
print("Making predictions on the test set...")
rf_predictions = rf_classifier.predict(test_embeddings)

# Evaluate the Random Forest model
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions)

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest F1-Score: {rf_f1:.4f}")

In [ ]:
df_t = pd.read_csv("transformer_predictions.csv")
df_w = pd.read_csv("bow_predictions.csv")
df_b = pd.read_csv("both_predictions.csv")

y_pred_t = df_t["predicted_label"].values
y_pred_w = df_w["predicted_label"].values
y_pred_b = df_b["predicted_label"].values
y_test = df_t["true_label"].values

In [ ]:
#### This is where the homework code goes.

def generate_data_for_mcnemar(y_pred1, y_pred2, y_test):

  neither = 0
  m1_only = 0
  m2_only = 0
  both = 0

  for i in range(len(y_test)):
    if y_pred1[i] == y_test[i]:
      if y_pred2[i] == y_test[i]:
        both += 1
      else:
        m1_only += 1
    else:
      if y_pred2[i] == y_test[i]:
        m2_only += 1
      else:
        neither += 1

  return [[neither, m1_only], [m2_only, both]]

In [ ]:
data = generate_data_for_mcnemar(y_pred_t, y_pred_w, y_test)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import chisquare

In [ ]:
res = mcnemar(data, exact=False)
alpha=0.05
print('Chi-squared statistic:',res.statistic)
print('Pvalue:',res.pvalue)
if res.pvalue < alpha:
    print('Models are different')
else:
    print('Models are same')

Chi-squared statistic: 5.78
Pvalue: 0.016209541409225422
Models are different


In [ ]:
def parse_file(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split(maxsplit=2)
            gold = int(parts[0])
            pred = int(parts[1])
            rows.append((gold, pred))
    return rows




In [ ]:
from collections import Counter

def extract_texts(path):
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split(maxsplit=2)
            if len(parts) == 3:
                texts.append(parts[2])
    return texts


# Load texts
transformer_texts = parse_file("/validation_predictions.txt")
bow_texts = parse_file("/rf_validation_predictions.txt")

# Compare as multisets
t_counter = Counter(transformer_texts)
b_counter = Counter(bow_texts)

# Differences
only_in_transformer = t_counter - b_counter
only_in_bow = b_counter - t_counter

num_diff_lines = sum(only_in_transformer.values()) + sum(only_in_bow.values())

print("Number of text lines that differ:", num_diff_lines)
#MAKE NOTE emojies dont seem to be present in the one it has

In [ ]:
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

def load_preds(file_path):
    """
    Load predictions from a text file.
    Expected format per line: <gold> <pred> <optional text>
    Splits on any whitespace (spaces, tabs) and only uses first two columns.
    Skips empty or malformed lines.
    """
    golds = []
    preds = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip empty lines
            parts = line.split(None, 2)  # split on any whitespace, max 2 splits
            if len(parts) < 2:
                continue  # skip malformed lines
            gold, pred = parts[:2]
            try:
                golds.append(int(gold))
                preds.append(int(pred))
            except ValueError:
                continue  # skip lines where conversion fails
    return golds, preds

# Load both prediction files
gold1, pred1 = load_preds("/validation_predictions.txt")
gold2, pred2 = load_preds("/rf_validation_predictions.txt")

# Align lengths
min_len = min(len(gold1), len(gold2))
if len(gold1) != len(gold2):
    print(f"Warning: mismatched lengths {len(gold1)} vs {len(gold2)}, trimming to {min_len}")
    gold1 = gold1[:min_len]
    pred1 = pred1[:min_len]
    gold2 = gold2[:min_len]
    pred2 = pred2[:min_len]

# Optional sanity check: all gold labels should match
assert all(g1 == g2 for g1, g2 in zip(gold1, gold2)), "Gold labels do not match!"

table = generate_data_for_mcnemar(pred1, pred2, gold1)

# Run McNemar test
res = mcnemar(table, exact=False)
alpha = 0.05

# Print results
print("Disagreements:")
print(f"Transformer correct, BoW wrong: {table[0][1]}")
print(f"BoW correct, Transformer wrong: {table[1][0]}")
print("\nMcNemar Test Results:")
print("Chi-squared statistic:", res.statistic)
print("P-value:", res.pvalue)
if res.pvalue < alpha:
    print("Models are significantly different")
else:
    print("No significant difference detected")
